# 랜덤신호론 10주차 실습
## 공분산과 상관 분석

1. 공분산과 상관계수의 의미를 설명한다.
2. 자기상관과 교차상관을 구분한다.
3. 주기신호와 잡음신호의 자기상관 차이를 비교한다.
4. lag 1 자기상관값을 AI 특징으로 추출한다.


## 0. 공통 환경 준비


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay, classification_report

rng=np.random.default_rng(2026)
plt.style.use('seaborn-v0_8-whitegrid')
FEATURE_NAMES=['mean','std','peak_to_peak','rms','autocorrelation','dominant_frequency','spectral_energy']

def lag1_autocorrelation(x):
    x=np.asarray(x,float)
    if len(x)<2 or np.std(x[:-1])==0 or np.std(x[1:])==0: return 0.0
    return float(np.corrcoef(x[:-1],x[1:])[0,1])

def extract_features(x,fs):
    x=np.asarray(x,float); centered=x-x.mean()
    spectrum=np.abs(np.fft.rfft(centered)); freqs=np.fft.rfftfreq(len(x),1/fs)
    return {'mean':float(x.mean()),'std':float(x.std()),'peak_to_peak':float(np.ptp(x)),
      'rms':float(np.sqrt(np.mean(x**2))),'autocorrelation':lag1_autocorrelation(x),
      'dominant_frequency':float(freqs[np.argmax(spectrum)]),
      'spectral_energy':float(np.sum(spectrum**2)/len(x))}

def generate_signal(kind,fs=50,seconds=4,seed=0):
    local=np.random.default_rng(seed); t=np.arange(0,seconds,1/fs)
    clean=np.sin(2*np.pi*3*t)+.35*np.sin(2*np.pi*7*t)
    if kind=='normal': x=clean+local.normal(0,.18,len(t))
    elif kind=='high_noise': x=clean+local.normal(0,.75,len(t))
    elif kind=='spike':
        x=clean+local.normal(0,.18,len(t)); idx=local.choice(len(t),4,False); x[idx]+=local.choice([-5,5],4)
    else: raise ValueError('kind 확인')
    return t,x

def build_dataset(n=40,fs=50):
    rows=[]; labels=['normal','high_noise','spike']
    for j,label in enumerate(labels):
        for i in range(n):
            _,x=generate_signal(label,fs=fs,seed=1000+100*j+i)
            row=extract_features(x,fs); row['label']=label; rows.append(row)
    return pd.DataFrame(rows)
print('준비 완료')


## 1. 공분산과 상관계수


In [ ]:
t=np.linspace(0,10,300)
x=np.sin(2*np.pi*.4*t)+rng.normal(0,.2,len(t))
y=1.8*x+rng.normal(0,.35,len(t)); z=-1.2*x+rng.normal(0,.35,len(t))
print('corr x,y:',np.corrcoef(x,y)[0,1]);print('corr x,z:',np.corrcoef(x,z)[0,1])
fig,ax=plt.subplots(1,2,figsize=(10,4));ax[0].scatter(x,y,s=10);ax[1].scatter(x,z,s=10,color='tomato');plt.show()


**관찰 1** 산점도의 방향과 상관계수 부호를 설명하세요.


## 2. 자기상관 비교


In [ ]:
fs=50;t=np.arange(0,4,1/fs);periodic=np.sin(2*np.pi*3*t);noise=rng.normal(0,1,len(t))
def ac(x):
    x=x-x.mean();r=np.correlate(x,x,mode='full')[len(x)-1:];return r/r[0]
rp,rn=ac(periodic),ac(noise);lags=np.arange(len(rp))/fs
plt.plot(lags[:100],rp[:100],label='periodic');plt.plot(lags[:100],rn[:100],label='noise');plt.legend();plt.show()
print('lag1:',lag1_autocorrelation(periodic),lag1_autocorrelation(noise))


**관찰 2** 두 자기상관 모양과 lag 1 값을 비교하세요.


## 3. 교차상관으로 지연 찾기


In [ ]:
delay=8;y2=np.roll(periodic,delay)
cc=np.correlate(y2-y2.mean(),periodic-periodic.mean(),mode='full')
lags=np.arange(-len(periodic)+1,len(periodic));best=lags[np.argmax(cc)]
print('추정 지연:',best,'표본,',best/fs,'초')


**미니 과제** 설정한 지연과 추정 지연을 비교하고, 상관만으로 인과를 증명할 수 없는 이유를 작성하세요.


## 참고 자료
- NumPy corrcoef: https://numpy.org/doc/stable/reference/generated/numpy.corrcoef.html
- NumPy correlate: https://numpy.org/doc/stable/reference/generated/numpy.correlate.html
